In [4]:
import pandas as pd  
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_absolute_error, r2_score  

df = pd.read_csv("delitos.csv", sep=";")

# quitamos espacios, pasamos a mayusculas y nos quedamos solo con mes y año 
# porque queremos analizar la evolucion temporal
df.columns = df.columns.str.strip().str.upper()
df = df[["AÑO", "MES"]]

# pasamos los meses a numeros para usar ml
mes_map = {"ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6, "JULIO": 7, "AGOSTO": 8, "SEPTIEMBRE": 9, "OCTUBRE": 10, "NOVIEMBRE": 11, "DICIEMBRE": 12}
df["MES"] = df["MES"].map(mes_map)
df = df.dropna()

# queremos saber delitos por mes asi que agrupamos por año y mes, 
# contamos cantidad de registros y ordenamos cronologicamente
df_g = df.groupby(["AÑO", "MES"]).size().reset_index(name="cantidad")
df_g = df_g.sort_values(["AÑO", "MES"]).reset_index(drop=True)

# creamos columnas guardando los delitos de los ultimos 3 meses
df_g["hace_1mes"] = df_g["cantidad"].shift(1)
df_g["hace_2mes"] = df_g["cantidad"].shift(2)
df_g["hace_3mes"] = df_g["cantidad"].shift(3)
df_g = df_g.dropna()

# no usamos año porque el modelo puede interpretar que los delitos aumentan con el paso del tiempo
X = df_g[["MES", "hace_1mes", "hace_2mes", "hace_3mes"]]
y = df_g["cantidad"]

# dividmos usando una relacion de 80% datos y 20% para test respetando el orden temporal
split = int(len(df_g) * 0.8)
X_train = X[:split]
X_test = X[split:]
y_train = y[:split]
y_test = y[split:]
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

#exportamos para dashboard
fechas = df_g.iloc[split:][["AÑO", "MES"]].copy()
resultado = pd.DataFrame({
    "Anio": fechas["AÑO"].astype(int).values,
    "Mes": fechas["MES"].astype(int).values,
    "Real": y_test.values.astype(int),
    "Prediccion": y_pred.round(0).astype(int)
})
resultado.to_csv("prediccion_delitos.csv", index=False)

# calculamos error
mae = round(mean_absolute_error(y_test, y_pred), 2)
r2 = round(r2_score(y_test, y_pred), 2)
print("MAE:", mae)
print("R2:", r2)

MAE: 654.09
R2: 0.27
